# Week 1 — Multi-Channel Signal Generation

This notebook creates reproducible synthetic ADC input data for the software-only phase of the project.

**Simulation:** 8 channels, 50 kHz sampling, 16-bit ±10 V ADC model.

The dataset contains clean tones, a chirp, harmonic distortion, Gaussian noise, a 10 kHz interference reference, and a transient.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dsp.signal_generator import AcquisitionConfig, generate_channels, quantize, snr_db

## 1. Acquisition configuration

In [ ]:
cfg = AcquisitionConfig()
t, clean, noisy, metadata = generate_channels(cfg)
quantized = quantize(noisy, cfg)

print(f'Sampling frequency : {cfg.fs/1000:.1f} kHz')
print(f'Duration           : {cfg.duration*1000:.1f} ms')
print(f'Samples/channel    : {cfg.num_samples}')
print(f'Channels           : {cfg.num_channels}')
print(f'ADC                : {cfg.adc_bits}-bit, {cfg.adc_min:g} to {cfg.adc_max:g} V')
print(f'LSB size           : {cfg.quantization_step*1e3:.4f} mV')
print(f'Configured noise   : {cfg.noise_rms*1e3:.3f} mV RMS')

## 2. Inspect the eight channels

In [ ]:
fig, axes = plt.subplots(8, 1, figsize=(12, 14), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(t * 1e3, noisy[:, i], linewidth=0.8)
    ax.set_ylabel(f'CH{i+1} [V]')
    ax.grid(True, alpha=0.25)
    ax.set_title(metadata[i]['description'], loc='left', fontsize=9)
axes[-1].set_xlabel('Time [ms]')
fig.suptitle('Synthetic 8-Channel ADC Inputs', y=1.01)
fig.tight_layout()
plt.show()

## 3. Time-domain comparison: clean, noisy and quantized

In [ ]:
ch = 1
window = t < 0.010
plt.figure(figsize=(12, 5))
plt.plot(t[window] * 1e3, clean[window, ch], label='Clean')
plt.plot(t[window] * 1e3, noisy[window, ch], label='Noisy', alpha=0.75)
plt.plot(t[window] * 1e3, quantized[window, ch], label='16-bit quantized', linewidth=1)
plt.xlabel('Time [ms]')
plt.ylabel('Voltage [V]')
plt.title('Channel 2: clean vs noisy vs quantized')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Frequency-domain analysis

In [ ]:
def spectrum(x, fs):
    n = len(x)
    freq = np.fft.rfftfreq(n, 1/fs)
    mag = np.abs(np.fft.rfft(x)) / n
    return freq, mag

for ch in [0, 1, 2, 3, 6]:
    f, mag = spectrum(noisy[:, ch], cfg.fs)
    plt.figure(figsize=(11, 3.5))
    plt.plot(f / 1000, 20*np.log10(np.maximum(mag, 1e-12)))
    plt.xlim(0, 15)
    plt.xlabel('Frequency [kHz]')
    plt.ylabel('Magnitude [dB]')
    plt.title(f'CH{ch+1} spectrum — {metadata[ch]["description"]}')
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 5. Basic measurements

In [ ]:
print('Channel | RMS clean [V] | RMS noise [mV] | SNR [dB]')
print('-' * 58)
for i in range(cfg.num_channels):
    rms_clean = np.sqrt(np.mean(clean[:, i]**2))
    noise = noisy[:, i] - clean[:, i]
    rms_noise = np.sqrt(np.mean(noise**2))
    snr = snr_db(clean[:, i], noisy[:, i])
    print(f'{i+1:7d} | {rms_clean:14.5f} | {rms_noise*1e3:14.5f} | {snr:8.2f}')

print('\nQuantization step:', f'{cfg.quantization_step*1e3:.5f} mV')
print('Ideal quantization RMS:', f'{cfg.quantization_step/np.sqrt(12)*1e3:.5f} mV')

## 6. Save a reproducible dataset

In [ ]:
import h5py

output_path = ROOT / 'data' / 'raw' / 'week1_synthetic_adc.h5'
with h5py.File(output_path, 'w') as h5:
    h5.create_dataset('time_s', data=t)
    h5.create_dataset('clean_v', data=clean)
    h5.create_dataset('noisy_v', data=noisy)
    h5.create_dataset('quantized_v', data=quantized)
    h5.attrs['fs_hz'] = cfg.fs
    h5.attrs['duration_s'] = cfg.duration
    h5.attrs['num_channels'] = cfg.num_channels
    h5.attrs['adc_bits'] = cfg.adc_bits
    h5.attrs['adc_min_v'] = cfg.adc_min
    h5.attrs['adc_max_v'] = cfg.adc_max
    h5.attrs['noise_rms_v'] = cfg.noise_rms
    h5.attrs['random_seed'] = cfg.seed

print(f'Saved: {output_path}')